# Trabalho 2 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências

In [1]:
%pip install nltk numpy pandas matplotlib "sentence-transformers==6.1.0" scikit-learn "transformers==5.20.1" torch "accelerate==1.15.0"


Note: you may need to restart the kernel to use updated packages.


## Pré-processamento

A representação lexical usa normalização para minúsculas, tokenização, remoção de stopwords em inglês e stemming Porter. O mesmo pré-processamento será aplicado aos documentos e às consultas.

In [2]:
# API do projeto: a implementação está organizada no pacote ri_project/.
from pathlib import Path
import pandas as pd

from ri_project.config import DATA_DIR, EXPERIMENTS_DIR, BM25_K1, BM25_B
from ri_project.data import load_dataset
from ri_project.preprocessing import tokenize, preprocess_tokens, preprocess
from ri_project.indexing import build_inverted_index
from ri_project.bm25 import BM25Retriever
from ri_project.typo import (
    build_kgram_index, character_kgrams, jaccard_coefficient, edit_distance,
    correct_query, TypoExperiment, KGRAM_SIZE, JACCARD_THRESHOLD,
    MAX_EDIT_DISTANCE,
)
from ri_project.semantic import SemanticRetriever
from ri_project.hybrid import HybridRRFRetriever, RRF_K, HYBRID_CANDIDATE_DEPTH
from ri_project.reranking import CrossEncoderReranker, HybridRerankingRetriever, RERANKER_MODEL_NAME, RERANKER_MODEL_REVISION
from ri_project.evaluation import run_evaluation, build_comparison_cases, build_reranking_analysis, tune_rrf_k


/home/arthur/repos/Trabalho-Modelos-Semanticos-RI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Leitura do NF-Corpus

Os arquivos JSONL são carregados como dicionários indexados pelo identificador do documento ou da consulta. Os três arquivos de qrels são lidos separadamente e também reunidos em um único DataFrame.

In [3]:
# O diretório de trabalho deve ser a raiz do projeto.
dataset = load_dataset(DATA_DIR)
documents = dataset['documents']
queries = dataset['queries']
qrels = dataset['qrels']
qrels_all = dataset['qrels_all']

print(f'Documentos: {len(documents):,}')
print(f'Consultas: {len(queries):,}')
print(f'Relevâncias (train/dev/test): {len(qrels_all):,}')
print('Colunas dos qrels:', list(qrels_all.columns))


Documentos: 3,633
Consultas: 3,237
Relevâncias (train/dev/test): 134,294
Colunas dos qrels: ['query-id', 'corpus-id', 'score']


## Construção do Índice Invertido

Cada documento é representado pela concatenação de título e texto. O índice associa cada termo processado aos documentos em que ocorre e à frequência do termo em cada documento. Também são armazenados o comprimento processado de cada documento e o total de documentos, usados no cálculo de BM25.

In [4]:
index_data = build_inverted_index(documents)
inverted_index = index_data['inverted_index']
document_lengths = index_data['document_lengths']
raw_vocabulary = index_data['raw_vocabulary']
num_documents = index_data['num_documents']
avg_document_length = index_data['avg_document_length']
num_postings = index_data['num_postings']
index_build_time_seconds = index_data['build_time_seconds']

def document_frequency(term):
    """Retorna em quantos documentos o termo processado aparece."""
    return len(inverted_index.get(term, {}))

print(f'Termos no vocabulário: {len(inverted_index):,}')
print(f'Total de postings: {num_postings:,}')
print(f'Tempo de construção do índice: {index_build_time_seconds:.4f} s')
print(f'Tamanho médio dos documentos: {avg_document_length:.2f} tokens')
print('Exemplo de postings:', next(iter(inverted_index.items())))


Termos no vocabulário: 19,054
Total de postings: 360,200
Tempo de construção do índice: 5.3003 s
Tamanho médio dos documentos: 165.82 tokens
Exemplo de postings: ('statin', {'MED-10': 12, 'MED-14': 6, 'MED-870': 1, 'MED-1193': 11, 'MED-1207': 2, 'MED-1258': 2, 'MED-1283': 1, 'MED-1429': 1, 'MED-1486': 3, 'MED-1554': 1, 'MED-1639': 1, 'MED-1874': 1, 'MED-1887': 2, 'MED-2428': 4, 'MED-2429': 8, 'MED-2431': 11, 'MED-2432': 1, 'MED-2525': 1, 'MED-3204': 1, 'MED-3430': 1, 'MED-4559': 9, 'MED-4562': 5, 'MED-4825': 1, 'MED-4827': 1, 'MED-4828': 8, 'MED-4829': 10, 'MED-4830': 8})


## Busca Lexical com BM25

A busca calcula pontuações apenas para documentos encontrados nas posting lists dos termos da consulta. O tempo médio pode ser medido sobre as consultas julgadas de uma partição, sem usar o resultado para ajustar parâmetros no conjunto de teste.

In [5]:
bm25 = BM25Retriever(index_data, k1=BM25_K1, b=BM25_B)
search_bm25 = bm25.search

bm25_timing = bm25.measure_query_time(queries, qrels, split='dev', top_k=100)
print('Configuração BM25:', {'k1': BM25_K1, 'b': BM25_B})
print(f"Tempo médio em dev (top-100): {bm25_timing['average_query_time_seconds']:.4f} s/consulta")
# Exemplo: search_bm25(queries['PLAIN-3']['text'], top_k=10)


Configuração BM25: {'k1': 1.2, 'b': 0.75}
Tempo médio em dev (top-100): 0.0008 s/consulta


## Tolerância a erros: k-gramas, Jaccard e distância de edição

O vocabulário de correção é formado pelos tokens do corpus antes de stemming, conforme o item 4 do PDF. Termos conhecidos são preservados; para termos fora do vocabulário, k-gramas geram candidatos, Jaccard filtra candidatos e distância de Levenshtein seleciona a alternativa mais próxima. O tamanho do k-grama e os limiares serão ajustados em `dev`, antes de aplicar a configuração às consultas de `test`.


In [6]:
from time import perf_counter

KGRAM_SIZE = 3
JACCARD_THRESHOLD = 0.30
MAX_EDIT_DISTANCE = 2
TYPO_OPERATIONS = ('insertion', 'deletion', 'substitution', 'transposition')
TYPO_LABELS = {
    'insertion': 'inserção', 'deletion': 'remoção',
    'substitution': 'substituição', 'transposition': 'troca de ordem',
}

kgram_index_build_start = perf_counter()
kgram_index = build_kgram_index(raw_vocabulary, KGRAM_SIZE)
kgram_index_build_time_seconds = perf_counter() - kgram_index_build_start
typo_experiment_runner = TypoExperiment(raw_vocabulary, queries, qrels, bm25)
print(f'Termos no vocabulário de correção (antes do stemming): {len(raw_vocabulary):,}')
print(f'Tempo de construção do índice de k-gramas: {kgram_index_build_time_seconds:.4f} s')
# Exemplo: correct_query(texto, raw_vocabulary, kgram_index)


Termos no vocabulário de correção (antes do stemming): 26,475
Tempo de construção do índice de k-gramas: 0.3151 s


## Experimento de tolerância a erros (item 4 do PDF)

O conjunto fixo de parâmetros será ajustado somente em `dev`, usando consultas e erros gerados sem depender da configuração avaliada. A configuração escolhida será então congelada para o experimento com exatamente 10 consultas de `test`, NDCG@10 original maior que zero e corrupções OOV. Serão comparadas as versões original, com erro e corrigida; os demais experimentos continuam usando as consultas originais.


In [7]:
# Ajusta apenas com dev; congela a configuração e avalia no conjunto test.
dev_tuning_results, best_typo_parameters = typo_experiment_runner.tune(
    n_queries=10, k=10
)
JACCARD_THRESHOLD = best_typo_parameters['jaccard_threshold']
MAX_EDIT_DISTANCE = best_typo_parameters['max_edit_distance']
KGRAM_SIZE = best_typo_parameters['kgram_size']
kgram_index = build_kgram_index(raw_vocabulary, KGRAM_SIZE)
print('Melhor configuração escolhida em dev:', best_typo_parameters)
dev_tuning_results.to_csv(EXPERIMENTS_DIR / 'typo_dev_tuning.csv', index=False)
display(dev_tuning_results.head(10))

typo_results = typo_experiment_runner.run(
    n_queries=10, split='test', k=10,
    jaccard_threshold=JACCARD_THRESHOLD,
    max_distance=MAX_EDIT_DISTANCE,
    kgram_size=KGRAM_SIZE,
)
typo_results.to_csv(EXPERIMENTS_DIR / 'typo_experiment.csv', index=False)
display(typo_results)


Melhor configuração escolhida em dev: {'kgram_size': 3, 'jaccard_threshold': 0.3, 'max_edit_distance': 2, 'mean_recovery_fraction': 0.48869656344442663, 'target_correction_accuracy': 0.8, 'mean_corrected_ndcg10': 0.3932222912276853, 'num_dev_cases': 10, 'num_cases_with_performance_loss': 9}


,kgram_size,jaccard_threshold,max_edit_distance,mean_recovery_fraction,target_correction_accuracy,mean_corrected_ndcg10,num_dev_cases,num_cases_with_performance_loss
0,3,0.3,2,0.488697,0.8,0.393222,10,9
1,3,0.2,2,0.488697,0.8,0.393222,10,9
2,2,0.5,2,0.488697,0.8,0.393222,10,9
3,2,0.4,2,0.488697,0.8,0.393222,10,9
4,2,0.3,2,0.488697,0.8,0.393222,10,9
5,4,0.3,2,0.488697,0.8,0.393222,10,9
6,2,0.2,2,0.488697,0.8,0.393222,10,9
7,4,0.2,2,0.488697,0.8,0.393222,10,9
8,3,0.3,1,0.662741,0.7,0.380049,10,9
9,3,0.2,1,0.662741,0.7,0.380049,10,9


,query_id,query_original,query_erro,query_corrigida,tipo_erro,token_original,token_com_erro,token_corrigido,correcoes_aplicadas,NDCG10_original,NDCG10_erro,NDCG10_corrigida,fracao_desempenho_recuperada,status_recuperacao
0,PLAIN-1018,DHA,bdha,bha,inserção,dha,bdha,bha,"[{""observed"": ""bdha"", ""replacement"": ""bha"", ""e...",0.521098,0.000000,0.000000,0.0,calculada
1,PLAIN-102,Stopping Heart Disease in Childhood,Stopping Heart Disease in hildhood,Stopping Heart Disease in childhood,remoção,childhood,hildhood,childhood,"[{""observed"": ""hildhood"", ""replacement"": ""chil...",0.125307,0.000000,0.125307,1.0,calculada
2,PLAIN-1039,domoic acid,aomoic acid,domoic acid,substituição,domoic,aomoic,domoic,"[{""observed"": ""aomoic"", ""replacement"": ""domoic...",1.000000,0.000000,1.000000,1.0,calculada
3,PLAIN-1088,ECMO,cemo,chemo,troca de ordem,ecmo,cemo,chemo,"[{""observed"": ""cemo"", ""replacement"": ""chemo"", ...",0.553146,0.000000,0.000000,0.0,calculada
4,PLAIN-1109,endocrine disruptors,endocrine adisruptors,endocrine disruptors,inserção,disruptors,adisruptors,disruptors,"[{""observed"": ""adisruptors"", ""replacement"": ""d...",0.469000,0.668306,0.469000,NaN,erro não reduziu o desempenho
5,PLAIN-112,Food Dyes and ADHD,Food Dyes and dhd,Food Dyes and adhd,remoção,adhd,dhd,adhd,"[{""observed"": ""dhd"", ""replacement"": ""adhd"", ""e...",0.402714,0.140743,0.402714,1.0,calculada
6,PLAIN-1130,ergothioneine,argothioneine,ergothioneine,substituição,ergothioneine,argothioneine,ergothioneine,"[{""observed"": ""argothioneine"", ""replacement"": ...",1.000000,0.000000,1.000000,1.0,calculada
7,PLAIN-1141,Evidence-based medicine,veidence-based medicine,evidence-based medicine,troca de ordem,evidence,veidence,evidence,"[{""observed"": ""veidence"", ""replacement"": ""evid...",0.220092,0.138862,0.220092,1.0,calculada
8,PLAIN-1151,factory farming practices,factory farming apractices,factory farming practices,inserção,practices,apractices,practices,"[{""observed"": ""apractices"", ""replacement"": ""pr...",0.571160,0.779908,0.571160,NaN,erro não reduziu o desempenho
9,PLAIN-1172,fenugreek,enugreek,fenugreek,remoção,fenugreek,enugreek,fenugreek,"[{""observed"": ""enugreek"", ""replacement"": ""fenu...",0.220092,0.000000,0.220092,1.0,calculada


## Recuperação semântica com sentence-BERT

Os documentos são codificados uma única vez com o bi-encoder pré-treinado `sentence-transformers/all-MiniLM-L6-v2`. Os embeddings normalizados e o alinhamento entre vetor e ID são persistidos em `Experiments/semantic/`. O cache só é reutilizado quando modelo, configuração e hash do texto da coleção conferem. Consultas usam o texto original (sem stemming lexical) e são ranqueadas por cosseno, calculado como produto interno entre vetores normalizados. A busca percorre todos os embeddings da coleção, adequada ao tamanho do NF-Corpus.

A primeira execução precisa baixar os pesos do modelo se eles não estiverem no cache local. O comprimento máximo efetivo e a quantidade de documentos truncados são medidos com o tokenizador do próprio modelo.

In [8]:
import sentence_transformers

semantic_retriever = SemanticRetriever(documents, queries, qrels)
search_semantic = semantic_retriever.search
semantic_document_ids = semantic_retriever.document_ids
semantic_document_embeddings = semantic_retriever.embeddings
semantic_model = semantic_retriever.model
semantic_device = semantic_retriever.device
semantic_max_seq_length = semantic_retriever.max_seq_length
semantic_truncated_documents = semantic_retriever.truncated_documents
semantic_embedding_build_seconds = semantic_retriever.embedding_build_seconds
semantic_cache_status = semantic_retriever.cache_status
semantic_dev_query_time_seconds = semantic_retriever.measure_query_time(split='dev', top_k=100)

print(f'Modelo: {semantic_retriever.model_name}')
print(f'Dispositivo: {semantic_device}; sentence-transformers {sentence_transformers.__version__}')
print(f'Documentos: {len(semantic_document_ids):,}; dimensão: {semantic_document_embeddings.shape[1]}')
print(f'Limite: {semantic_max_seq_length} tokens; documentos truncados: {semantic_truncated_documents:,}')
print(f'Embeddings normalizados: {semantic_retriever.normalize}; similaridade: cosseno')
print(f'Estado do cache: {semantic_cache_status}')
if semantic_embedding_build_seconds is not None:
    print(f'Tempo registrado para geração dos embeddings: {semantic_embedding_build_seconds:.2f} s')
print(f'Arquivos: {semantic_retriever.embeddings_path}, {semantic_retriever.doc_ids_path}, {semantic_retriever.manifest_path}')
print('Exemplo de busca:', search_semantic(queries[next(iter(queries))].get('text', ''), top_k=5))
print(f'Tempo médio de consulta em dev (encode + busca top-100): {semantic_dev_query_time_seconds:.4f} s/consulta')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1097.67it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (386 > 256). Running this sequence through the model will result in indexing errors
Batches: 100%|██████████| 114/114 [00:09<00:00, 12.33it/s]


Modelo: sentence-transformers/all-MiniLM-L6-v2
Dispositivo: cuda; sentence-transformers 6.1.0
Documentos: 3,633; dimensão: 384
Limite: 256 tokens; documentos truncados: 2,864
Embeddings normalizados: True; similaridade: cosseno
Estado do cache: gerado e salvo
Tempo registrado para geração dos embeddings: 9.27 s
Arquivos: /home/arthur/repos/Trabalho-Modelos-Semanticos-RI/Experiments/semantic/all_minilm_l6_v2_document_embeddings.npy, /home/arthur/repos/Trabalho-Modelos-Semanticos-RI/Experiments/semantic/all_minilm_l6_v2_document_ids.json, /home/arthur/repos/Trabalho-Modelos-Semanticos-RI/Experiments/semantic/all_minilm_l6_v2_manifest.json
Exemplo de busca: [{'doc_id': 'MED-2439', 'score': 0.6983506679534912}, {'doc_id': 'MED-2434', 'score': 0.6489325761795044}, {'doc_id': 'MED-2440', 'score': 0.6331394910812378}, {'doc_id': 'MED-2427', 'score': 0.6134428381919861}, {'doc_id': 'MED-2102', 'score': 0.542138397693634}]
Tempo médio de consulta em dev (encode + busca top-100): 0.0192 s/consul

## Busca híbrida com Reciprocal Rank Fusion (RRF)

A fusão combina os rankings de BM25 e sentence-BERT por posição, sem misturar diretamente suas escalas de score.

In [9]:
# Ajuste obrigatório: comparar k=10, 60 e 100 exclusivamente em dev.
rrf_dev_tuning, rrf_dev_tuning_per_query, best_rrf_k = tune_rrf_k(
    queries, qrels, bm25, semantic_retriever, documents.keys(),
    output_dir=EXPERIMENTS_DIR, split='dev',
    k_values=(10, 60, 100), candidate_depth=100, cutoff=10,
)
RRF_K_SELECTED = best_rrf_k
print('Resultados do ajuste RRF em dev:')
print(rrf_dev_tuning.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f'k selecionado por NDCG@10 médio em dev: {RRF_K_SELECTED}')
display(rrf_dev_tuning)

# Congela k selecionado para a avaliação final em test.
hybrid_retriever = HybridRRFRetriever(
    bm25, semantic_retriever, k=RRF_K_SELECTED,
    candidate_depth=100, document_ids=documents.keys(),
)
search_hybrid = hybrid_retriever.search
print('Exemplo de busca híbrida:', search_hybrid(
    queries[next(iter(queries))].get('text', ''), top_k=5
))


Resultados do ajuste RRF em dev:
  k split  query_count  NDCG10  candidate_depth_per_method
 10   dev          324  0.3246                         100
100   dev          324  0.3193                         100
 60   dev          324  0.3187                         100
k selecionado por NDCG@10 médio em dev: 10


,k,split,query_count,NDCG10,candidate_depth_per_method
0,10,dev,324,0.324595,100
1,100,dev,324,0.319253,100
2,60,dev,324,0.318746,100


Exemplo de busca híbrida: [{'doc_id': 'MED-2434', 'score': 0.17424242424242425, 'component_ranks': {'lexical_bm25': 1, 'semantic_all-MiniLM-L6-v2': 2}}, {'doc_id': 'MED-2427', 'score': 0.15476190476190477, 'component_ranks': {'lexical_bm25': 2, 'semantic_all-MiniLM-L6-v2': 4}}, {'doc_id': 'MED-2439', 'score': 0.14090909090909093, 'component_ranks': {'lexical_bm25': 10, 'semantic_all-MiniLM-L6-v2': 1}}, {'doc_id': 'MED-2774', 'score': 0.125, 'component_ranks': {'lexical_bm25': 6, 'semantic_all-MiniLM-L6-v2': 6}}, {'doc_id': 'MED-2102', 'score': 0.10114942528735632, 'component_ranks': {'lexical_bm25': 19, 'semantic_all-MiniLM-L6-v2': 5}}]


## Reranking do Top-50 híbrido (item 9)

O cross-encoder recebe cada par consulta-documento e reordena somente as 50 primeiras posições do ranking híbrido. A cauda mantém a ordem original. O modelo é avaliado em `dev` antes da avaliação final em `test`.

In [10]:
reranker = CrossEncoderReranker(
    model_name=RERANKER_MODEL_NAME, revision=RERANKER_MODEL_REVISION,
    rerank_depth=50, batch_size=32,
)
reranked_retriever = HybridRerankingRetriever(
    hybrid_retriever, reranker, documents
)

# Executa em dev para avaliar o modelo escolhido antes de congelá-lo para test.
RERANKING_DEV_DIR = EXPERIMENTS_DIR / 'reranking_dev'
reranking_dev_result = run_evaluation(
    documents, queries, qrels, bm25, semantic_retriever,
    hybrid=hybrid_retriever, reranked=reranked_retriever,
    output_dir=RERANKING_DEV_DIR, split='dev', cutoff=10, ranking_depth=100,
)
print(reranking_dev_result['metrics_summary'].to_string(
    index=False, float_format=lambda value: f'{value:.4f}'
))
print('Manifesto observado em dev:', reranker.manifest())
reranker.save_manifest(EXPERIMENTS_DIR / 'reranker_dev_manifest.json')
reranker.reset_statistics()


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 391.38it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


split                     model  query_count    P10    R10  MAP10  NDCG10  average_query_time_seconds  ranking_depth  evaluation_cutoff  average_retrieval_time_seconds  average_reranking_time_seconds
  dev              lexical_bm25          324 0.2364 0.1292 0.0929  0.2970                      0.0005            100                 10                          0.0005                          0.0000
  dev semantic_all-MiniLM-L6-v2          324 0.2401 0.1364 0.0947  0.3024                      0.0156            100                 10                          0.0156                          0.0000
  dev                hybrid_rrf          324 0.2583 0.1458 0.1027  0.3246                      0.0148            100                 10                          0.0148                          0.0000
  dev          hybrid_reranking          324 0.2623 0.1551 0.1084  0.3319                      0.2506            100                 10                          0.0133                          0.2363


## Avaliação lexical, semântica e híbrida


In [11]:
evaluation_result = run_evaluation(
    documents, queries, qrels, bm25, semantic_retriever,
    hybrid=hybrid_retriever, reranked=reranked_retriever,
    output_dir=EXPERIMENTS_DIR, split='test', cutoff=10, ranking_depth=100,
)
metrics_per_query = evaluation_result['metrics_per_query']
metrics_summary = evaluation_result['metrics_summary']
rankings_by_query = evaluation_result['rankings_by_query']
evaluation_judgments = evaluation_result['judgments']
evaluation_query_ids = evaluation_result['query_ids']
rankings_path = evaluation_result['ranking_paths']

print(f'Partição: test; consultas avaliadas por método: {len(evaluation_query_ids)}')
print('Configuração lexical congelada:', {'model': 'BM25', 'k1': BM25_K1, 'b': BM25_B})
print('Configuração híbrida congelada:', {
    'method': 'RRF', 'k': hybrid_retriever.k,
    'candidate_depth_per_method': hybrid_retriever.candidate_depth,
})
print('Configuração do reranker congelada:', {
    'model': reranker.model_name, 'revision': reranker.revision,
    'rerank_depth': reranker.rerank_depth,
    'max_input_tokens': reranker.max_input_tokens,
})
print('Configuração semântica congelada:', {
    'model': semantic_retriever.model_name,
    'dimension': int(semantic_document_embeddings.shape[1]), 'similarity': 'cosine',
})
print(metrics_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print('Rankings salvos:', ', '.join(str(path) for path in rankings_path.values()))
print('Resumo de métricas: lexical, semântico, híbrido e híbrido + reranking.')
reranker.save_manifest(EXPERIMENTS_DIR / 'reranker_manifest.json')
print('Manifesto do cross-encoder:', EXPERIMENTS_DIR / 'reranker_manifest.json')
print(f'Métricas por consulta salvas: {EXPERIMENTS_DIR / "metrics_per_query.csv"}')


Partição: test; consultas avaliadas por método: 323
Configuração lexical congelada: {'model': 'BM25', 'k1': 1.2, 'b': 0.75}
Configuração híbrida congelada: {'method': 'RRF', 'k': 10, 'candidate_depth_per_method': 100}
Configuração do reranker congelada: {'model': 'cross-encoder/ms-marco-MiniLM-L6-v2', 'revision': '233902d25c440f23af6f7d6e94d2946bac0bee0a', 'rerank_depth': 50, 'max_input_tokens': 512}
Configuração semântica congelada: {'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384, 'similarity': 'cosine'}
split                     model  query_count    P10    R10  MAP10  NDCG10  average_query_time_seconds  ranking_depth  evaluation_cutoff  average_retrieval_time_seconds  average_reranking_time_seconds
 test              lexical_bm25          323 0.2390 0.1532 0.1218  0.3278                      0.0006            100                 10                          0.0006                          0.0000
 test semantic_all-MiniLM-L6-v2          323 0.2433 0.1550 0.1108  0

### Comparação por consulta e casos representativos

A célula seguinte seleciona os três maiores ganhos de NDCG@10 de cada sistema e salva o Top5 de cada método com julgamentos e evidência lexical observada.

In [12]:
comparison_cases, comparison_top5 = build_comparison_cases(
    documents, queries, evaluation_result, output_dir=EXPERIMENTS_DIR
)
print('Seis casos de comparação salvos:')
print(comparison_cases[[
    'query_id', 'winner_by_ndcg10', 'delta_ndcg10_semantic_minus_lexical', 'query',
]].to_string(index=False))
print(f'Detalhes Top5 com scores e relevância: {EXPERIMENTS_DIR / "comparison_cases_top5.csv"}')


Seis casos de comparação salvos:
  query_id          winner_by_ndcg10  delta_ndcg10_semantic_minus_lexical                                                     query
 PLAIN-307 semantic_all-MiniLM-L6-v2                             0.671386 Vitamin D: Shedding some light on the new recommendations
PLAIN-3085 semantic_all-MiniLM-L6-v2                             0.570642  The Difficulty of Arriving at a Vitamin D Recommendation
PLAIN-1537 semantic_all-MiniLM-L6-v2                             0.567043                                            low-carb diets
PLAIN-1473              lexical_bm25                            -1.000000                                                  kohlrabi
PLAIN-2081              lexical_bm25                            -1.000000                                                shelf life
PLAIN-1288              lexical_bm25                            -0.760079                                                    grapes
Detalhes Top5 com scores e relevância: /hom

In [13]:
def print_comparison_top5(cases, heading):
    """Imprime Top5 de ambos os métodos com os graus de relevância dos qrels."""
    print('=' * 88)
    print(heading)
    print('=' * 88)
    labels = {'lexical_bm25': 'BM25 (lexical)',
              'semantic_all-MiniLM-L6-v2': 'all-MiniLM-L6-v2 (semântico)'}
    for case in cases.itertuples(index=False):
        print(f"\n{case.query_id} | consulta: {case.query}")
        print(f'NDCG@10 BM25={case.ndcg10_lexical:.4f} | semântico={case.ndcg10_semantic:.4f} | Δ semântico−BM25={case.delta_ndcg10_semantic_minus_lexical:+.4f}')
        for method in ('lexical_bm25', 'semantic_all-MiniLM-L6-v2'):
            print(f'  Top 5 — {labels[method]}:')
            docs = comparison_top5.loc[(comparison_top5.query_id == case.query_id) &
                (comparison_top5.method == method)].sort_values('rank')
            for row in docs.itertuples(index=False):
                grade = float(row.relevance)
                status = f'RELEVANTE, grau {grade:g}' if grade > 0 else 'NÃO RELEVANTE, grau 0'
                print(f'    {int(row.rank)}. [{status}] {row.doc_id} | score={row.score:.4f} | {row.title}')

semantic_cases_to_print = comparison_cases.loc[
    comparison_cases['winner_by_ndcg10'] == 'semantic_all-MiniLM-L6-v2'
].head(3)
print_comparison_top5(semantic_cases_to_print,
    'TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO SEMÂNTICO')


TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO SEMÂNTICO

PLAIN-307 | consulta: Vitamin D: Shedding some light on the new recommendations
NDCG@10 BM25=0.1480 | semântico=0.8194 | Δ semântico−BM25=+0.6714
  Top 5 — BM25 (lexical):
    1. [NÃO RELEVANTE, grau 0] MED-3099 | score=15.8256 | The aryl hydrocarbon receptor and its xenobiotic ligands: a fundamental trigger for cardiovascular diseases.
    2. [NÃO RELEVANTE, grau 0] MED-4872 | score=12.8617 | Safety considerations and potential interactions of vitamins: should vitamins be considered drugs?
    3. [NÃO RELEVANTE, grau 0] MED-3373 | score=12.4809 | The Influence of Social Involvement, Neighborhood Aesthetics, and Community Garden Participation on Fruit and Vegetable Consumption
    4. [NÃO RELEVANTE, grau 0] MED-1579 | score=10.6282 | Crohn's disease: a review of treatment options and current research.
    5. [NÃO RELEVANTE, grau 0] MED-1777 | score=10.3658 | Shedding Light on the Controversy Surrounding the Temporal Decline in Huma

In [14]:
lexical_cases_to_print = comparison_cases.loc[
    comparison_cases['winner_by_ndcg10'] == 'lexical_bm25'
].head(3)
print_comparison_top5(lexical_cases_to_print,
    'TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO LEXICAL (BM25)')


TRÊS CONSULTAS COM MAIOR VANTAGEM DO MODELO LEXICAL (BM25)

PLAIN-1473 | consulta: kohlrabi
NDCG@10 BM25=1.0000 | semântico=0.0000 | Δ semântico−BM25=-1.0000
  Top 5 — BM25 (lexical):
    1. [RELEVANTE, grau 1] MED-4455 | score=11.6606 | Glucoraphanin and 4-hydroxyglucobrassicin contents in seeds of 59 cultivars of broccoli, raab, kohlrabi, radish, cauliflower, brussels sprouts, kal...
    2. [NÃO RELEVANTE, grau 0] MED-10 | score=0.0000 | Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
    3. [NÃO RELEVANTE, grau 0] MED-1000 | score=0.0000 | Neurobehavioral function and low-level exposure to brominated flame retardants in adolescents: a cross-sectional study
    4. [NÃO RELEVANTE, grau 0] MED-1002 | score=0.0000 | Polybrominated diphenyl ethers (PBDEs), hydroxylated PBDEs (OH-PBDEs), and measures of thyroid function in second trimester pregnant women in California
    5. [NÃO RELEVANTE, grau 0] MED-1003 | score=0.0000 | In Utero and Childhood Polybrominat

## Análise por consulta do reranking (item 10)

A análise compara NDCG@10 do híbrido RRF e do híbrido após reranking nas mesmas consultas de `test`. São selecionados os dois maiores ganhos, o caso com delta mais próximo de zero (desempate pelo menor número de documentos cuja mudança de posição afeta o Top-10) e a maior piora. Os CSVs registram Top-10 antes/depois e documentos cuja mudança de posição afeta a janela avaliada; as tabelas desta seção são regeneradas na execução e as decisões metodológicas estão em `Notas_Projeto_RI.md`.


In [15]:
reranking_analysis = build_reranking_analysis(
    documents, queries, evaluation_result, output_dir=EXPERIMENTS_DIR
)
print('Casos selecionados:')
display(reranking_analysis['cases'])
print('Documentos que mudaram de posição no Top-10:')
display(reranking_analysis['movements'])
print('Top-10 antes/depois:')
display(reranking_analysis['top10'])
print('Artefatos: reranking_analysis_cases.csv, reranking_analysis_movements.csv e reranking_analysis_top10.csv')


Casos selecionados:


,case,query_id,query,ndcg10_hybrid,ndcg10_reranking,delta_ndcg10,changed_documents_in_top10_union
0,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,0.000000,0.561693,0.561693,17
1,segundo maior ganho,PLAIN-1473,kohlrabi,0.630930,1.000000,0.369070,18
2,pouca ou nenhuma alteração,PLAIN-1557,magnesium,0.315163,0.315163,0.000000,8
3,maior piora,PLAIN-2540,Does Cholesterol Size Matter?,0.615336,0.303078,-0.312258,16


Documentos que mudaram de posição no Top-10:


,case,query_id,query,doc_id,hybrid_rank,reranked_rank,rank_change,relevance,hybrid_score,reranker_score,title,text
9,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-2760,11,1,-10,1.0,0.088828,0.755206,Multivitamins in the Prevention of Cancer in M...,Context Multivitamin preparations are the most...
1,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-4929,7,2,-5,0.0,0.097418,0.177749,Vitamins E and C in the Prevention of Cardiova...,Context Basic and observational studies sugges...
3,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-2758,12,3,-9,1.0,0.087500,-2.012956,Multivitamins in the Prevention of Cardiovascu...,Context Though multivitamins aim to prevent vi...
0,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-1986,3,4,1,0.0,0.114719,-4.050281,Long-term morbidity and mortality of overweigh...,BACKGROUND: Overweight in adults is associated...
4,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-4398,18,5,-13,0.0,0.071429,-4.215020,High school dietary dairy intake and teenage a...,BACKGROUND: Previous studies suggest possible ...
14,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-2761,19,6,-13,1.0,0.069462,-5.364428,Who uses multivitamins? A cross-sectional stud...,PURPOSE: The aim of this study was to examine ...
5,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-3932,23,7,-16,0.0,0.062500,-6.234940,Caffeine and risk of Parkinson disease in a la...,Background Caffeine consumption has been assoc...
13,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-5124,47,8,-39,0.0,0.028571,-7.139714,Egg Consumption and Cardiovascular Disease and...,Background Reduction in dietary cholesterol is...
15,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-761,5,9,4,0.0,0.103741,-7.343164,The counseling practices of internists.,OBJECTIVES: To determine the counseling practi...
16,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,MED-1622,31,10,-21,0.0,0.043478,-7.523326,"Coffee, caffeine, and risk of completed suicid...",Objective To evaluate the association between ...


Top-10 antes/depois:


,case,query_id,query,method,rank,doc_id,relevance,score,title
10,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,hybrid_reranking,1,MED-2760,1.0,0.755206,Multivitamins in the Prevention of Cancer in M...
11,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,hybrid_reranking,2,MED-4929,0.0,0.177749,Vitamins E and C in the Prevention of Cardiova...
12,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,hybrid_reranking,3,MED-2758,1.0,-2.012956,Multivitamins in the Prevention of Cardiovascu...
13,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,hybrid_reranking,4,MED-1986,0.0,-4.050281,Long-term morbidity and mortality of overweigh...
14,maior ganho,PLAIN-1320,Harvard Physicians’ Study II,hybrid_reranking,5,MED-4398,0.0,-4.215020,High school dietary dairy intake and teenage a...
...,...,...,...,...,...,...,...,...,...
25,segundo maior ganho,PLAIN-1473,kohlrabi,hybrid_rrf,6,MED-3441,0.0,0.076923,Marine edible algae as disease preventers.
26,segundo maior ganho,PLAIN-1473,kohlrabi,hybrid_rrf,7,MED-1002,0.0,0.071429,"Polybrominated diphenyl ethers (PBDEs), hydrox..."
27,segundo maior ganho,PLAIN-1473,kohlrabi,hybrid_rrf,8,MED-3447,0.0,0.071429,Seaweed prevents breast cancer?
28,segundo maior ganho,PLAIN-1473,kohlrabi,hybrid_rrf,9,MED-1003,0.0,0.066667,In Utero and Childhood Polybrominated Diphenyl...


Artefatos: reranking_analysis_cases.csv, reranking_analysis_movements.csv e reranking_analysis_top10.csv


## Geração com RAG (item 11)

O fluxo usa `reranked_retriever`: consulta original → BM25 + semântica → RRF com k escolhido em dev → cross-encoder no Top-50 → Top-5 → LLM → resposta.
Cada documento do corpus é uma unidade de contexto, com título, texto integral e `doc_id`. Não há chunking. O prompt exige evidências, citações `[doc_id]` e declaração de evidência insuficiente. A resposta é gerada em inglês, idioma da coleção.

Escolha inicial de implementação: `Qwen/Qwen2.5-1.5B-Instruct`, local, CPU ou CUDA, geração greedy (temperatura configurada 0), seed 42 e até 512 novos tokens. Essa escolha não resulta de comparação no test. A primeira execução baixa os pesos, se necessário. O manifesto registra a revisão efetivamente carregada, versões, hardware e parâmetros; reutilize essa revisão para repetir o experimento. Se o prompt integral mais a saída reservada exceder a janela do modelo, a execução falha explicitamente sem cortar fontes.

As células abaixo executam uma demonstração em **dev**, sem seleção pelos qrels de relevância, e salvam resposta, fontes, prompt e configuração em `Experiments/rag/`. O experimento pareado com as dez perguntas de test está na seção seguinte; sua avaliação humana e análise de falhas (itens 13–14) ficam para a próxima etapa.


In [16]:
from ri_project.rag import RAGConfig, LocalCausalGenerator, RAGPipeline

rag_config = RAGConfig(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    revision='989aa7980e4cf806f80c7fef2b1adb7bc71aa306',  # Pesos fixados.
    temperature=0.0, max_new_tokens=512, seed=42,
    language='English', context_documents=5, retrieval_depth=100,
)
rag_generator = LocalCausalGenerator(rag_config)
rag_pipeline = RAGPipeline(reranked_retriever, documents, rag_generator)
print('Configuração da LLM:', rag_generator.manifest())


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 189.77it/s]


Configuração da LLM: {'config': {'model_name': 'Qwen/Qwen2.5-1.5B-Instruct', 'revision': '989aa7980e4cf806f80c7fef2b1adb7bc71aa306', 'temperature': 0.0, 'max_new_tokens': 512, 'seed': 42, 'language': 'English', 'context_documents': 5, 'retrieval_depth': 100}, 'execution': 'local', 'resolved_model_revision': None, 'transformers_version': '5.20.1', 'torch_version': '2.14.1+cu130', 'python_version': '3.14.4', 'device': 'cuda', 'cpu': '13th Gen Intel(R) Core(TM) i5-13420H', 'gpu': 'NVIDIA GeForce RTX 4050 Laptop GPU', 'dtype': 'torch.bfloat16', 'context_window_tokens': 32768, 'model_generation_defaults': {'max_length': None, 'max_new_tokens': None, 'min_length': None, 'min_new_tokens': None, 'early_stopping': None, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': None, 'use_mtp': None, 'use_cache': None, 'cache_implementation': None, 'cache_config': None, 'max_cache_len': None, 'temperature': 0.7, 'top_k': 20, 'top_p': 0.8, 'min_p': None, 'top_h': None, 'typical_p': 

In [17]:
# Exemplo determinístico em dev: não escolhe casos favoráveis em test.
rag_query_id = sorted(set(qrels['dev']['query-id'].astype(str)))[0]
rag_question = queries[rag_query_id]['text']
rag_output_path = EXPERIMENTS_DIR / 'rag' / f'{rag_query_id}.json'
rag_result = rag_pipeline.answer(
    rag_question, query_id=rag_query_id, output_path=rag_output_path,
)
print(f'{rag_query_id}: {rag_question}')
print(rag_result['answer'])
display(pd.DataFrame(rag_result['contexts'])[['rank', 'doc_id', 'title', 'score']])
print('Tokens de entrada/saída:', rag_result['input_tokens'], rag_result['output_tokens'])
print('Tempo total (s):', rag_result['total_seconds'])
print('Resposta, contexto integral, prompt e manifesto:', rag_output_path)


PLAIN-1: Why Deep Fried Foods May Cause Cancer
Based on the provided documents, deep-fried foods may cause cancer primarily because they involve high-heat cooking methods which can lead to the formation of potentially harmful compounds such as aldehydes, acrolein, heterocyclic amines, polycyclic aromatic hydrocarbons, and acrylamide. These substances are known to have carcinogenic properties according to the evidence from various studies mentioned in the documents. For instance:

1. **"Consumption of deep-fried foods and risk of prostate cancer"** states that deep frying involves the addition of oil/fat, which increases the risk of prostate cancer.
   
2. **"Intake of fried meat and risk of cancer: a follow-up study in Finland"** mentions that mutagens in fried meat may contribute to cancer.

3. **"Olive, soybean and palm oils intake have a similar acute detrimental effect over the endothelial function in healthy young subjects"** notes that deep-frying processes alter the chemical com

,rank,doc_id,title,score
0,1,MED-2418,Consumption of deep-fried foods and risk of pr...,7.015487
1,2,MED-4070,Intake of fried meat and risk of cancer: a fol...,3.864536
2,3,MED-5269,"Olive, soybean and palm oils intake have a sim...",-0.015304
3,4,MED-4818,Pork intake and human papillomavirus-related d...,-0.131995
4,5,MED-5033,Cancer is a Preventable Disease that Requires ...,-0.469526


Tokens de entrada/saída: 1877 292
Tempo total (s): 17.678361416999905
Resposta, contexto integral, prompt e manifesto: /home/arthur/repos/Trabalho-Modelos-Semanticos-RI/Experiments/rag/PLAIN-1.json


## Experimento sem RAG × com RAG (item 12)

A seleção reproduz o sorteio oficial: seed 42 entre as 42 perguntas de test, conferindo os dez IDs do enunciado. A recebe somente a pergunta e instrução de idioma, sem acesso aos documentos. B recebe a mesma pergunta e os cinco documentos integrais após reranking. Ambas usam a mesma instância da LLM, inglês e os mesmos parâmetros; cada chamada tem histórico independente.

A execução salva vinte checkpoints e arquivos `answers.jsonl`, `answers.csv`, `comparison.md`, `rag_experiment.csv` (formato do item 15, notas humanas pendentes) e `run_status.json` em `Experiments/rag_experiment/`. A retomada verifica a identidade de configuração, contextos e prompts antes de reutilizar uma resposta. Os resultados não constituem a avaliação humana dos itens 13–14.



In [18]:
import hashlib

from ri_project.rag import select_context
from ri_project.rag_experiment import select_experiment_queries, run_paired_experiment

rag_test_ids = select_experiment_queries(queries, qrels['test']['query-id'])
rag_cases = []
for qid in rag_test_ids:
    query = queries[qid]['text']
    ranking = reranked_retriever.search(query, top_k=100)
    rag_cases.append({
        'query_id': qid, 'query': query,
        'contexts': select_context(ranking, documents, count=5),
        'reranked_top100': ranking,
    })
rag_bundle = {
    'protocol': 'assignment_item_12', 'split': 'test',
    'sampling_seed': 42, 'question_candidate_count': 42,
    'query_ids': rag_test_ids, 'cases': rag_cases,
    'dataset_sha256': {
        name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
        for name in ('corpus.jsonl', 'queries.jsonl', 'qrels/dev.tsv', 'qrels/test.tsv')
    },
    'retrieval': {
        'rrf_k': hybrid_retriever.k, 'candidate_depth': 100,
        'semantic': semantic_retriever.manifest,
        'reranker': reranker.manifest(),
        'context_documents': 5, 'context_truncation': False,
    },
}
RAG_EXPERIMENT_DIR = EXPERIMENTS_DIR / 'rag_experiment'
RAG_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
import json
(RAG_EXPERIMENT_DIR / 'contexts.json').write_text(
    json.dumps(rag_bundle, ensure_ascii=False, indent=2), encoding='utf-8',
)
rag_paired_answers = run_paired_experiment(rag_bundle, rag_generator, RAG_EXPERIMENT_DIR)
display(pd.DataFrame(rag_paired_answers)[[
    'query_id', 'condition', 'answer', 'input_tokens', 'output_tokens', 'generation_seconds',
]])


Gerando PLAIN-68: A_without_rag
Concluídas 1/20; saída: 238 tokens.
Gerando PLAIN-68: B_with_rag
Concluídas 2/20; saída: 93 tokens.
Gerando PLAIN-2450: A_without_rag
Concluídas 3/20; saída: 138 tokens.
Gerando PLAIN-2450: B_with_rag
Concluídas 4/20; saída: 245 tokens.
Gerando PLAIN-153: A_without_rag
Concluídas 5/20; saída: 334 tokens.
Gerando PLAIN-153: B_with_rag
Concluídas 6/20; saída: 319 tokens.
Gerando PLAIN-2850: A_without_rag
Concluídas 7/20; saída: 320 tokens.
Gerando PLAIN-2850: B_with_rag
Concluídas 8/20; saída: 154 tokens.
Gerando PLAIN-2750: A_without_rag
Concluídas 9/20; saída: 113 tokens.
Gerando PLAIN-2750: B_with_rag
Concluídas 10/20; saída: 186 tokens.
Gerando PLAIN-270: A_without_rag
Concluídas 11/20; saída: 199 tokens.
Gerando PLAIN-270: B_with_rag
Concluídas 12/20; saída: 77 tokens.
Gerando PLAIN-2470: A_without_rag
Concluídas 13/20; saída: 87 tokens.
Gerando PLAIN-2470: B_with_rag
Concluídas 14/20; saída: 231 tokens.
Gerando PLAIN-2430: A_without_rag
Concluídas 15

,query_id,condition,answer,input_tokens,output_tokens,generation_seconds
0,PLAIN-68,A_without_rag,Chicken nuggets are typically made from chicke...,24,238,9.391752
1,PLAIN-68,B_with_rag,"According to the document titled ""The autopsy ...",2076,93,4.197359
2,PLAIN-2450,A_without_rag,Organic foods are produced without the use of ...,23,138,5.506868
3,PLAIN-2450,B_with_rag,"Yes, based on the provided documents, organic ...",1894,245,11.529572
4,PLAIN-153,A_without_rag,Probiotics are beneficial bacteria that can he...,25,334,22.146414
5,PLAIN-153,B_with_rag,Based on the evidence provided:\n\n1. **Meal T...,2482,319,22.047203
6,PLAIN-2850,A_without_rag,The choice between white and dark meat when it...,27,320,18.073221
7,PLAIN-2850,B_with_rag,Based on the information provided in Document ...,2742,154,5.407269
8,PLAIN-2750,A_without_rag,There is no scientific evidence to support the...,27,113,3.651686
9,PLAIN-2750,B_with_rag,"Yes, according to the document titled ""Randomi...",2979,186,8.241733


## Avaliação do RAG (item 13)

O contexto é exatamente o Top-5 enviado à LLM nas dez consultas oficiais. Relevância binária é `grau > 0`; Recall@5 usa todos os relevantes nos qrels, e as médias são macro. A avaliação da geração exige notas humanas 0–2 e justificativas. O preparo cria fichas embaralhadas e um caderno com os mesmos documentos de referência para avaliar A e B. Na consolidação, rastreabilidade de A é N/A e fica fora da média. Campos vazios são pendências, sem conversão para zero.


In [19]:
from pathlib import Path
from ri_project.rag_evaluation import prepare_rag_evaluation, aggregate_human_evaluation

RAG_EVALUATION_DIR = Path('Experiments/rag_experiment/evaluation')
rag_evaluation = prepare_rag_evaluation(
    experiment_dir='Experiments/rag_experiment', data_dir='nfcorpus/nfcorpus',
    rankings_dir='Experiments', output_dir=RAG_EVALUATION_DIR,
)
display(rag_evaluation['context_metrics'])
display(rag_evaluation['context_summary'])
print('Caderno para avaliação humana:', RAG_EVALUATION_DIR / 'human_review.md')
print('Preencha ratings_reviewer_1.csv e, opcionalmente, ratings_reviewer_2.csv sem consultar blind_key.json.')


ValueError: Respostas/contextos/seed mudaram. Use outra pasta de avaliação para preservar as notas.

In [ ]:
# As fichas são criadas vazias. Reexecute após os integrantes preencherem suas notas.
# Se houver dois avaliadores completos, escolha explicitamente o avaliador final.
FINAL_REVIEWER = None
rating_files = [RAG_EVALUATION_DIR / f'ratings_reviewer_{i}.csv' for i in (1, 2)]
human_evaluation = aggregate_human_evaluation(
    rating_files, experiment_dir='Experiments/rag_experiment',
    data_dir='nfcorpus/nfcorpus', output_dir=RAG_EVALUATION_DIR,
    final_reviewer=FINAL_REVIEWER,
)
print('Estado da avaliação humana:', human_evaluation['status'])
display(human_evaluation['summary'])
display(human_evaluation['agreement'])
print('Discussão das médias:', RAG_EVALUATION_DIR / 'human_evaluation_report.md')


Estado da avaliação humana: {'status': 'pending_human_ratings', 'artifact_fingerprint': 'acbe9244b7c18304fcdc6ca47d238c3f1e064eb7f516965d88f60f98bb0a816c', 'final_reviewer': None, 'ratings_fingerprint': 'b5e063f5927f69c8cb587a4c4b963b7ac8b1ed7e1e1263060a472c1d9e6252ae', 'ratings_csv_sha256': 'f46fc2442a2f9ccd1585cb8003b83247d4db764dd52ec210835285e2d76dc9a7', 'complete_reviewers': [], 'required_cells_per_reviewer': 70, 'rated_cells': 0, 'quality_scores_created_automatically': False}


,evaluator_id,condition,criterion,applicable,expected_count,rated_count,mean,complete
0,avaliador_1,A,fidelidade,True,10,0,None,False
1,avaliador_1,A,completude,True,10,0,None,False
2,avaliador_1,A,utilidade,True,10,0,None,False
3,avaliador_1,A,rastreabilidade,False,0,0,None,True
4,avaliador_1,B,fidelidade,True,10,0,None,False
5,avaliador_1,B,completude,True,10,0,None,False
6,avaliador_1,B,utilidade,True,10,0,None,False
7,avaliador_1,B,rastreabilidade,True,10,0,None,False
8,avaliador_2,A,fidelidade,True,10,0,None,False
9,avaliador_2,A,completude,True,10,0,None,False


,evaluator_a,evaluator_b,condition,criterion,n_pairs,exact_agreement,quadratic_kappa
0,avaliador_1,avaliador_2,A,fidelidade,0,None,None
1,avaliador_1,avaliador_2,A,completude,0,None,None
2,avaliador_1,avaliador_2,A,utilidade,0,None,None
3,avaliador_1,avaliador_2,B,fidelidade,0,None,None
4,avaliador_1,avaliador_2,B,completude,0,None,None
5,avaliador_1,avaliador_2,B,utilidade,0,None,None
6,avaliador_1,avaliador_2,B,rastreabilidade,0,None,None


Discussão das médias: Experiments/rag_experiment/evaluation/human_evaluation_report.md


## Análise de falhas ponta a ponta (item 14)

Os dossiers das dez consultas incluem respostas A/B, documentos integrais e posições no lexical, semântico, RRF e reranking. Os qrels servem ao diagnóstico e não modificam o contexto. A chegada de um documento relevante não prova que uma afirmação específica foi sustentada.

Após consolidar as notas humanas, o código sugere casos usando a diferença da média de Fidelidade, Completude e Utilidade (pesos iguais); rastreabilidade fica fora desse delta porque não se aplica a A. Maior ganho positivo sugere melhora, menor delta não positivo sugere ausência de ajuda, e menor delta absoluto de até 1/3, com no máximo um ponto de mudança total entre os três critérios, sugere pouca diferença, com consultas distintas. Essa é uma convenção de auxílio, não uma regra do enunciado. Categorias ausentes ficam explícitas. Confirme ou ajuste os casos em `case_reviews.json`, com motivo da seleção, comparação textual, estágio investigado e trechos literais de evidência. O relatório final só é exportado após validar notas e revisões humanas completas.


In [20]:
stage_diagnostics = rag_evaluation['diagnostics']
display(stage_diagnostics['summary'][[
    'query_id', 'lexical_relevant_top100', 'semantic_relevant_top100',
    'relevant_in_candidate_union', 'relevant_in_hybrid_top50',
    'relevant_in_hybrid_top5', 'relevant_in_context',
]])
print('Posições e textos de documentos:', RAG_EVALUATION_DIR / 'stage_document_evidence.csv')
print('Dossiers de todas as consultas:', RAG_EVALUATION_DIR / 'failure_analysis_dossiers.md')
print('Sugestão após notas humanas:', RAG_EVALUATION_DIR / 'failure_case_selection.json')
print('Revisões a preencher:', RAG_EVALUATION_DIR / 'case_reviews.json')


NameError: name 'rag_evaluation' is not defined

In [ ]:
from ri_project.rag_analysis import finalize_failure_analysis

analysis_result = finalize_failure_analysis(
    experiment_dir='Experiments/rag_experiment', data_dir='nfcorpus/nfcorpus',
    output_dir=RAG_EVALUATION_DIR,
)
print('Resultado/estado da análise:', analysis_result)
if analysis_result.suffix == '.json':
    print(analysis_result.read_text(encoding='utf-8'))
else:
    from IPython.display import Markdown
    display(Markdown(analysis_result.read_text(encoding='utf-8')))


Resultado/estado da análise: Experiments/rag_experiment/evaluation/failure_analysis_status.json
{
  "status": "pending_human_ratings",
  "artifact_fingerprint": "acbe9244b7c18304fcdc6ca47d238c3f1e064eb7f516965d88f60f98bb0a816c",
  "automatic_stage_diagnostics_complete": true
}

